# 🏗️ 3D Asset Factory
### 🛠️ Developed by: **Arafat Ahmed Mubin**
### 🌐 Brand: **2M**
---


### Text-to-3D Generation with Professional Mesh Optimization

This notebook provides a complete pipeline for generating game-ready 3D assets from text. 

**The Workflow:**
1. **Concept Art:** FLUX.1-schnell generates a high-quality character or object concept.
2. **3D Reconstruction:** TripoSR transforms the 2D concept into a 3D mesh in seconds.
3. **Optimization:** Mesh decimation and background removal ensure the asset is lightweight.
4. **Export:** Industry-standard .glb and .obj downloads for Unity, Unreal, or Roblox.

## 🛠️ Step 1: 3D Graphics Setup
Install the TripoSR engine, background removal tools, and 3D processing libraries.

In [ ]:
# Install 3D and Image backends
!pip install -q -U triposr rembg trimesh gradio diffusers accelerate transformers

## 🧠 Step 2: Asset Engine Configuration
We load the FLUX model for concept generation and the TripoSR model for 3D reconstruction.

In [ ]:
import torch
import gc
import numpy as np
from PIL import Image
from diffusers import FluxPipeline
from triposr.model import TripoSR
from triposr.utils import remove_background, resize_foreground
import trimesh
import gradio as gr
from rembg import remove

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
flux_pipe = None
tripo_model = None

def clear_memory():
    gc.collect()
    torch.cuda.empty_cache()

def load_flux():
    global flux_pipe
    if flux_pipe is None:
        print("Loading FLUX concept engine...")
        flux_pipe = FluxPipeline.from_pretrained("black-forest-labs/FLUX.1-schnell", torch_dtype=torch.float16)
        flux_pipe.enable_model_cpu_offload()
    return flux_pipe

def load_tripo():
    global tripo_model
    if tripo_model is None:
        print("Loading TripoSR reconstruction engine...")
        tripo_model = TripoSR.from_pretrained("stabilityai/TripoSR")
        tripo_model.to(DEVICE)
    return tripo_model

## 🎛️ Step 3: Production Pipeline Logic
This includes the background removal, 3D generation, and mesh optimization (decimation).

In [ ]:
def factory_process(prompt, decimation_ratio):
    clear_memory()
    
    # 1. Concept Generation (Flux)
    f_pipe = load_flux()
    concept_prompt = f"{prompt}, character sheet, isolated on white background, high quality, 3d game asset style"
    image = f_pipe(concept_prompt, num_inference_steps=4, width=512, height=512).images[0]
    
    # 2. Background Removal
    image_no_bg = remove(image)
    
    # 3. 3D Reconstruction
    t_model = load_tripo()
    with torch.no_grad():
        results = t_model(image_no_bg, device=DEVICE)
    
    # 4. Mesh Optimization
    mesh = results[0]
    if decimation_ratio < 1.0:
        target_faces = int(len(mesh.faces) * decimation_ratio)
        mesh = mesh.simplify_quadratic_decimation(target_faces)
    
    # 5. Export
    glb_path = "asset_output.glb"
    mesh.export(glb_path)
    return image, glb_path

## 🏗️ Step 4: Asset Factory Dashboard
Launch the interactive UI to generate and download your 3D models.

In [ ]:
with gr.Blocks(theme=gr.themes.Soft()) as factory:
    gr.Markdown("# 🏗️ 3D Asset Factory")
    
    with gr.Row():
        with gr.Column(scale=1):
            prompt_in = gr.Textbox(label="Asset Prompt", placeholder="A robotic knight in silver armor, detailed, sci-fi...")
            decimate = gr.Slider(0.1, 1.0, value=0.5, step=0.1, label="Mesh Density (Lower = More Optimized)")
            gen_btn = gr.Button("Forge Asset", variant="primary")
            
        with gr.Column(scale=2):
            with gr.Row():
                img_out = gr.Image(label="Generated Concept")
                mesh_out = gr.Model3D(label="Interactive 3D Preview")
            download_btn = gr.File(label="Download Game-Ready GLB")
    
    def run_factory(prompt, ratio):
        img, glb = factory_process(prompt, ratio)
        return img, glb, glb

    gen_btn.click(run_factory, [prompt_in, decimate], [img_out, mesh_out, download_btn])

factory.launch(share=True, debug=True)

---
**© 2026 2M | All Rights Reserved**
*Powered by the 2M Ecosystem (2M Dev's, 2M Future Facts, 2M Business Blogs)*